# 🎯 Clustering: Agrupación No Supervisada

**Módulo 03** | **Sesión 6** | **Duración estimada: 1h**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/notebooks/03_05_clustering.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Aprendizaje no supervisado | Entender la diferencia entre clasificación (supervisado) y clustering (no supervisado) | Comprensión |
| 2 | K-Means | Aplicar el algoritmo K-Means para agrupar datos | Aplicación |
| 3 | Selección de K | Usar el método del codo y silhouette score para elegir el número de clusters | Análisis |
| 4 | Perfilamiento | Interpretar y nombrar los clusters encontrados | Evaluación |
| 5 | Comunicación | Traducir los resultados del clustering en recomendaciones de gestión | Síntesis |

## 💡 ¿Por qué es importante para tu práctica docente?

Hasta ahora hemos trabajado con **aprendizaje supervisado**: teníamos datos con respuestas conocidas (etiquetas) y entrenábamos modelos para predecirlas.

Pero muchas preguntas interesantes no tienen respuestas predefinidas:

- ¿Existen **perfiles naturales** entre los docentes de FACES? (¿Quiénes son similares entre sí?)
- ¿Se pueden agrupar los estudiantes por sus **patrones de rendimiento**?
- ¿Hay **segmentos** diferenciados entre los empleados según salario, antigüedad y desempeño?

El **clustering** (agrupamiento) descubre estos grupos de forma automática, sin que le digamos cuáles son. Esto es especialmente útil para:

- **Diseñar intervenciones personalizadas** (un grupo puede necesitar tutoría, otro mentoría, otro nada)
- **Comunicar hallazgos** a directivos ("identificamos 3 perfiles docentes: ...")
- **Descubrir patrones ocultos** que no habíamos considerado

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## 📖 Sección 1: ¿Qué es clustering?

### Supervisado vs No Supervisado

| Aspecto | Supervisado | No Supervisado |
|---------|-------------|----------------|
| **Datos** | Tienen etiquetas (respuestas) | No tienen etiquetas |
| **Objetivo** | Predecir la etiqueta | Descubrir estructura oculta |
| **Ejemplo** | "Este estudiante está en riesgo" | "Hay 3 grupos de estudiantes con patrones similares" |
| **Algoritmos** | Regresión, clasificación | Clustering, reducción de dimensionalidad |

### ¿Cómo funciona el clustering?

El clustering busca grupos de observaciones que sean:
- **Similares dentro del grupo** (cohesión)
- **Diferentes entre grupos** (separación)

### Ejemplos cotidianos

- **Supermercado**: agrupar clientes por patrones de compra → diseñar ofertas personalizadas
- **Universidad**: agrupar docentes por perfil → diseñar programas de desarrollo diferenciados
- **Salud**: agrupar pacientes por síntomas → identificar subtipos de una enfermedad

---

## 🔄 Sección 2: K-Means

**K-Means** es el algoritmo de clustering más popular por su simplicidad.

### ¿Cómo funciona? (simplificado)

1. **Elige K** = número de grupos que quieres encontrar
2. **Coloca K centros** al azar en el espacio de datos
3. **Asigna** cada punto al centro más cercano
4. **Mueve** cada centro al promedio de sus puntos asignados
5. **Repite** los pasos 3-4 hasta que los centros no se muevan más

### Analogía

Imagina que tienes que organizar mesas en un evento con 100 invitados:
1. Colocas 4 mesas en el salón
2. Cada invitado se sienta en la mesa más cercana
3. Mueves cada mesa al centro de sus invitados
4. Los invitados cambian de mesa si hay una más cercana
5. Repites hasta que nadie cambia de mesa

---

## 🔧 Sección 3: Preparar Datos

Trabajaremos con la **encuesta docente** de FACES para encontrar perfiles de docentes.

In [ ]:
# Cargar datos de la encuesta docente
df = pd.read_csv('../../datasets/universidad/encuesta_docente.csv')

print(f'Docentes: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head()

In [ ]:
# Seleccionar las variables numéricas para clustering
features_cluster = [
    'satisfaccion_general', 'satisfaccion_salarial', 
    'satisfaccion_infraestructura', 'satisfaccion_desarrollo',
    'publicaciones', 'carga_horaria', 'antiguedad'
]

X_cluster = df[features_cluster].copy()

# Verificar datos faltantes
print('Datos faltantes por columna:')
print(X_cluster.isnull().sum())
print(f'\nTotal de filas completas: {X_cluster.dropna().shape[0]} de {len(X_cluster)}')

# Eliminar filas con datos faltantes (si hay)
X_cluster = X_cluster.dropna()
df_clean = df.loc[X_cluster.index].copy()

In [ ]:
# Estandarizar las variables
# IMPORTANTE: K-Means es sensible a la escala
# Si no estandarizamos, las variables con valores grandes (como carga_horaria)
# dominarán sobre las que tienen valores pequeños (como satisfacción 1-5)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Verificar: las variables estandarizadas tienen media~0 y std~1
X_scaled_df = pd.DataFrame(X_scaled, columns=features_cluster)
print('Estadísticas después de estandarizar:')
print(X_scaled_df.describe().round(2).loc[['mean', 'std']])
print('\n(Todas las variables ahora están en la misma escala)')

---

## 📐 Sección 4: Elegir K — Método del Codo

El parámetro más importante de K-Means es **K** (cuántos grupos queremos). ¿Cómo elegirlo?

### Método del Codo (Elbow Method)

Ejecutamos K-Means con K=2, 3, 4, ..., 8 y medimos la **inercia** (qué tan compactos son los clusters). Buscamos el punto donde agregar más clusters ya no mejora significativamente — el "codo" de la curva.

In [ ]:
# Calcular inercia para diferentes valores de K
inercias = []
rango_k = range(2, 9)

for k in rango_k:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inercias.append(kmeans.inertia_)

# Gráfico del codo
plt.figure(figsize=(10, 5))
plt.plot(rango_k, inercias, 'o-', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (suma de distancias al centro)')
plt.title('Método del Codo — ¿Cuántos clusters?')
plt.xticks(rango_k)
plt.tight_layout()
plt.show()

print('Busca el "codo": el punto donde la curva deja de bajar rápidamente')
print('Ese K ofrece buen balance entre complejidad y calidad de agrupamiento')

---

## 📏 Sección 5: Silhouette Score

El **silhouette score** (coeficiente de silueta) es otra forma de evaluar la calidad del clustering:

- Mide qué tan bien cada punto encaja en su cluster comparado con los otros clusters
- Varía de **-1 a 1**:
  - **Cercano a 1**: el punto está bien asignado
  - **Cercano a 0**: el punto está en la frontera entre dos clusters
  - **Negativo**: el punto probablemente está en el cluster equivocado

In [ ]:
# Calcular silhouette score para diferentes K
silhouettes = []

for k in rango_k:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f'K={k}: Silhouette Score = {sil:.4f}')

# Gráfico
plt.figure(figsize=(10, 5))
plt.plot(rango_k, silhouettes, 's-', color='coral', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score por Número de Clusters')
plt.xticks(rango_k)
plt.tight_layout()
plt.show()

mejor_k = rango_k[np.argmax(silhouettes)]
print(f'\nMejor K según silhouette: {mejor_k} (score = {max(silhouettes):.4f})')

---

## 🎯 Sección 6: Aplicar K-Means

Con el K seleccionado, ajustemos el modelo final y exploremos los clusters.

In [ ]:
# Aplicar K-Means con el K elegido
# Usaremos K=3 como punto de partida (ajustar según los resultados del codo/silhouette)
k_elegido = 3

kmeans_final = KMeans(n_clusters=k_elegido, random_state=42, n_init=10)
df_clean['cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'Distribución de docentes por cluster (K={k_elegido}):')
print(df_clean['cluster'].value_counts().sort_index())
print(f'\nSilhouette Score: {silhouette_score(X_scaled, df_clean["cluster"]):.4f}')

In [ ]:
# Explorar los clusters: promedio de cada variable por grupo
perfil_clusters = df_clean.groupby('cluster')[features_cluster].mean().round(2)

print('Perfil promedio de cada cluster:')
perfil_clusters

---

## 📊 Sección 7: Visualizar Clusters

Como tenemos más de 2 variables, no podemos visualizar directamente los clusters. Usaremos **PCA** (Análisis de Componentes Principales) para reducir a 2 dimensiones y poder graficar.

In [ ]:
# Reducir a 2 dimensiones con PCA para visualizar
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                      c=df_clean['cluster'], cmap='Set1', 
                      alpha=0.6, s=60, edgecolors='white')

# Marcar los centroides
centroides_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1], 
            c='black', marker='X', s=200, linewidths=2, label='Centroides')

plt.xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
plt.ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
plt.title(f'Clusters de Docentes FACES (K={k_elegido})')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Comparar clusters con gráfico de barras para cada variable
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feature in enumerate(features_cluster):
    ax = axes[i]
    medias = df_clean.groupby('cluster')[feature].mean()
    colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:k_elegido]
    medias.plot(kind='bar', ax=ax, color=colores)
    ax.set_title(feature, fontsize=11)
    ax.set_xlabel('')
    ax.set_xticklabels([f'C{c}' for c in range(k_elegido)], rotation=0)

# Ocultar el último subplot si sobra
if len(features_cluster) < len(axes):
    axes[-1].set_visible(False)

plt.suptitle('Comparación de Variables por Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 🏷️ Sección 8: Interpretar y Nombrar Clusters

La parte más importante del clustering no es el algoritmo, sino la **interpretación**. Debemos dar un nombre significativo a cada grupo basándonos en sus características.

In [ ]:
# Tabla de perfiles detallada
perfil = df_clean.groupby('cluster')[features_cluster].agg(['mean', 'std']).round(2)

# Simplificar: solo medias
perfil_simple = df_clean.groupby('cluster')[features_cluster].mean().round(2)
perfil_simple['n_docentes'] = df_clean.groupby('cluster').size()

# Agregar información categórica
for col in ['departamento', 'dedicacion', 'formacion']:
    moda_por_cluster = df_clean.groupby('cluster')[col].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A')
    perfil_simple[f'{col}_mas_comun'] = moda_por_cluster

print('Perfil completo de cada cluster:')
perfil_simple.T

In [ ]:
# Nombrar los clusters basándose en los perfiles
# (Estos nombres deben ajustarse según los resultados reales)

print('='*70)
print('PERFILES DOCENTES IDENTIFICADOS POR EL MODELO')
print('='*70)

for cluster_id in sorted(df_clean['cluster'].unique()):
    datos = df_clean[df_clean['cluster'] == cluster_id]
    medias = datos[features_cluster].mean()
    n = len(datos)
    
    # Determinar características dominantes
    sat_promedio = (medias['satisfaccion_general'] + medias['satisfaccion_salarial'] + 
                    medias['satisfaccion_infraestructura'] + medias['satisfaccion_desarrollo']) / 4
    
    print(f'\nCluster {cluster_id} ({n} docentes):')
    print(f'  Satisfacción promedio: {sat_promedio:.1f}/5')
    print(f'  Publicaciones promedio: {medias["publicaciones"]:.1f}')
    print(f'  Carga horaria promedio: {medias["carga_horaria"]:.0f}h')
    print(f'  Antigüedad promedio: {medias["antiguedad"]:.0f} años')
    
    # Sugerencia de nombre basada en el perfil
    if sat_promedio >= 3.5 and medias['publicaciones'] > 4:
        nombre = 'Docentes experimentados y satisfechos'
    elif sat_promedio < 2.5:
        nombre = 'Docentes en riesgo de burnout'
    elif medias['publicaciones'] > 5 and medias['antiguedad'] < 10:
        nombre = 'Docentes jóvenes y productivos'
    elif medias['carga_horaria'] > 20:
        nombre = 'Docentes con alta carga laboral'
    else:
        nombre = 'Docentes de perfil intermedio'
    
    print(f'  Nombre sugerido: "{nombre}"')

---

## 💼 Sección 9: Caso — Perfiles de Empleados

Apliquemos clustering al dataset de recursos humanos para identificar perfiles de empleados.

In [ ]:
# Cargar datos de RRHH
rrhh = pd.read_csv('../../datasets/empresarial/rrhh_nomina.csv')
print(f'Empleados: {len(rrhh)}')
rrhh.head()

In [ ]:
# Seleccionar y estandarizar variables
features_rrhh = ['salario_mensual_usd', 'evaluacion_desempeno', 'ausencias_anuales', 'antiguedad']
X_rrhh = rrhh[features_rrhh].dropna()

scaler_rrhh = StandardScaler()
X_rrhh_scaled = scaler_rrhh.fit_transform(X_rrhh)

# K-Means con K=3
kmeans_rrhh = KMeans(n_clusters=3, random_state=42, n_init=10)
rrhh_clean = rrhh.loc[X_rrhh.index].copy()
rrhh_clean['cluster'] = kmeans_rrhh.fit_predict(X_rrhh_scaled)

print('Distribución de empleados por cluster:')
print(rrhh_clean['cluster'].value_counts().sort_index())

In [ ]:
# Perfil de cada cluster de empleados
perfil_rrhh = rrhh_clean.groupby('cluster')[features_rrhh].mean().round(2)
perfil_rrhh['n_empleados'] = rrhh_clean.groupby('cluster').size()

print('Perfil promedio de cada cluster de empleados:')
perfil_rrhh.T

In [ ]:
# Visualizar los clusters de empleados
pca_rrhh = PCA(n_components=2)
X_rrhh_pca = pca_rrhh.fit_transform(X_rrhh_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_rrhh_pca[:, 0], X_rrhh_pca[:, 1], 
                      c=rrhh_clean['cluster'], cmap='Set2', 
                      alpha=0.6, s=50, edgecolors='white')
plt.xlabel(f'Componente 1 ({pca_rrhh.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'Componente 2 ({pca_rrhh.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Clusters de Empleados (K=3)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

---

## ✏️ Ejercicios

### Ejercicio 1: Clustering de estudiantes

Usando `rendimiento_academico.csv`, agrupa a los estudiantes según `promedio`, `asistencia_pct` y `materias_aprobadas`.

1. Carga los datos y selecciona las 3 variables
2. Estandariza con StandardScaler
3. Aplica K-Means con K=3
4. Calcula el perfil promedio de cada cluster
5. Asigna un nombre descriptivo a cada grupo (ej: "Estudiantes destacados", "En riesgo", etc.)

In [ ]:
# Ejercicio 1: Clustering de estudiantes
# Tu código aquí


### Ejercicio 2: Elegir el mejor K

Usando los datos de `encuesta_docente.csv`:

1. Calcula el silhouette score para K=2, 3, 4, 5, 6, 7, 8
2. ¿Cuál K da el mejor silhouette score?
3. ¿Coincide con lo que sugiere el método del codo?

In [ ]:
# Ejercicio 2: Elegir el mejor K
# Tu código aquí


### Ejercicio 3: Recomendación de política

Basándote en el análisis de clusters de docentes (Sección 8):

1. Elige el cluster que consideres más "en riesgo" (baja satisfacción, alta carga, etc.)
2. Escribe un párrafo con una recomendación de política de RRHH para atender a ese grupo
3. ¿Qué datos adicionales te gustaría tener para mejorar el análisis?

In [ ]:
# Ejercicio 3: Recomendación de política
# Tu código aquí

# Escribe tu recomendación como comentarios:
# Cluster elegido: 
# Recomendación: 
# Datos adicionales que necesitaría: 


---

## 📋 Resumen

| Concepto | Punto clave |
|----------|-------------|
| **Clustering** | Encuentra grupos naturales SIN etiquetas previas (no supervisado) |
| **K-Means** | Asigna puntos al centro más cercano, mueve centros, repite |
| **Estandarización** | Indispensable: igualar la escala de todas las variables |
| **Método del codo** | Elegir K donde la inercia deja de bajar significativamente |
| **Silhouette score** | Mide calidad del clustering (-1 a 1, mayor es mejor) |
| **PCA** | Reduce dimensiones para poder visualizar clusters en 2D |
| **Interpretación** | Lo más importante: nombrar y describir cada cluster en contexto |

## 📚 Referencias

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer. Capítulo 12: Unsupervised Learning.

2. Scikit-learn developers. (2024). *Clustering*. https://scikit-learn.org/stable/modules/clustering.html

3. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Capítulo 9: Unsupervised Learning Techniques.

4. Jain, A. K. (2010). *Data clustering: 50 years beyond K-means*. Pattern Recognition Letters, 31(8), 651-666.

5. Rousseeuw, P. J. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*. Journal of Computational and Applied Mathematics, 20, 53-65.